In [33]:
import importlib
import subprocess

import Advanced_SQL as asql
import duckdb as duckdb
import SQL as sql

_ = importlib.reload(asql)
_ = importlib.reload(sql)

Use this cell to test any sql queries. This can be done in one of the following ways:

1. Fill in the `qry` within the """ """.
2. Call the question functions that you have already filled in from the SQL.py and Advanced_SQL.py files. An example of question_1 being called from the SQL.py file is currently commented out.

This cell can be copied if you wish to run multiple queries.


In [45]:
qry = """
    WITH repayment_totals AS (
        SELECT
            CustomerID,
            COUNT(*) AS TotalRepayments
        FROM repayments
        GROUP BY CustomerID
    ),
    
    customer_categories AS (
        SELECT
            c.CustomerID,
            c.Age,
            c.CorrectedAge,
            c.Gender,
            COALESCE(r.TotalRepayments, 0) AS TotalRepayments,
            CASE
                WHEN c.CorrectedAge < 20 THEN 'Teen'
                WHEN c.CorrectedAge < 30 THEN 'Young Adult'
                WHEN c.CorrectedAge < 60 THEN 'Adult'
                ELSE 'Pensioner'
            END AS AgeCategory
        FROM corrected_customers c
        LEFT JOIN repayment_totals r
            ON c.CustomerID = r.CustomerID
    )
    
    SELECT
        CustomerID,
        Age,
        CorrectedAge,
        Gender,
        AgeCategory,
        DENSE_RANK() OVER (
            PARTITION BY AgeCategory
            ORDER BY TotalRepayments DESC
        ) AS Rank
    
    FROM customer_categories
    
    ORDER BY CustomerID;
   
"""


# qry = sql.question_3()


with duckdb.connect("database/loan.db") as cursor:
    result_set = cursor.execute(qry).df()

# with duckdb.connect("database/loan.db") as con:
#     con.execute("""
#         COPY corrected_customers
#         TO 'corrected_customers.csv'
#         (HEADER, DELIMITER ',');
#     """)

# print("CSV exported successfully!")

result_set

,CustomerID,Age,CorrectedAge,Gender,AgeCategory,Rank
0,1,24,71,Female,Pensioner,7
1,2,34,71,Female,Pensioner,8
2,3,48,24,Female,Young Adult,3
3,4,34,34,Female,Adult,8
4,5,67,48,Female,Adult,10
...,...,...,...,...,...,...
1009,997,65,41,Female,Adult,11
1010,998,52,39,Female,Adult,10
1011,999,71,52,Female,Adult,10
1012,999,71,65,Female,Pensioner,9


Should you break/incorrectly update any database tables, the database can be reset by running the following cell


In [45]:
load_script_path = "database/database_load.py"
subprocess.run(["python3", load_script_path])

CompletedProcess(args=['python3', 'database/database_load.py'], returncode=0)